# 02. Data Cleaning, Sentinel Fixes & Temporal Splits
**Objective**: Handle missing pricing data, forward-fill corporate action intervals, and implement strict chronological partitioning (70% Train, 15% Validation, 15% Test) to prevent data leakage.

In [ ]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

stocks_df = pd.read_parquet("../data/sp500_universe.parquet")
macro_df = pd.read_parquet("../data/macro_indicators.parquet")

stocks_df['Date'] = pd.to_datetime(stocks_df['Date'])
macro_df['Date'] = pd.to_datetime(macro_df['Date'])

# Sequential clean per stock
def clean_series(df):
    cleaned = []
    for ticker, group in df.groupby('Ticker'):
        group = group.sort_values('Date').reset_index(drop=True)
        if len(group) < 252:
            continue
        group = group.ffill().bfill()
        cleaned.append(group)
    return pd.concat(cleaned, ignore_index=True)

cleaned_stocks = clean_series(stocks_df)
print(f"Cleaned Records: {len(cleaned_stocks):,} across {cleaned_stocks['Ticker'].nunique()} tickers.")

## 1. Establishing Chronological Boundaries

In [ ]:
dates = np.sort(cleaned_stocks['Date'].unique())
train_cutoff = dates[int(len(dates) * 0.70)]
val_cutoff = dates[int(len(dates) * 0.85)]

train_set = cleaned_stocks[cleaned_stocks['Date'] < train_cutoff]
val_set = cleaned_stocks[(cleaned_stocks['Date'] >= train_cutoff) & (cleaned_stocks['Date'] < val_cutoff)]
test_set = cleaned_stocks[cleaned_stocks['Date'] >= val_cutoff]

print(f"Train Set: {train_set['Date'].min().date()} to {train_set['Date'].max().date()} ({len(train_set):,} rows)")
print(f"Val Set:   {val_set['Date'].min().date()} to {val_set['Date'].max().date()} ({len(val_set):,} rows)")
print(f"Test Set:  {test_set['Date'].min().date()} to {test_set['Date'].max().date()} ({len(test_set):,} rows)")